# Camada Silver — `ecommerce_categorias` 

Lê a Bronze física em `az://squad1/bronze/ecommerce_categorias`, aplica as 10 regras de qualidade, grava **somente linhas válidas** na Silver física em `az://squad1/silver/ecommerce_categorias` e registra as falhas em `az://squad1/dq_monitoring_logs`.

Este notebook possui modo de reprocessamento para quando os arquivos já foram lidos anteriormente.

In [0]:
# MAGIC %run ../../utils/utils

##  Imports e parâmetros

In [0]:


#  Carrega as funções utilitárias (gravar_delta, ler_delta, etc)


import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone
from functools import reduce

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_categorias"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - categorias - Run ID: {RUN_ID}")

## Leitura do Micro-lote e Tabelas de Referência (Joins)

In [0]:
# =================================================================================
# 1. LEITURA DA BRONZE E ISOLAMENTO DO MICRO-LOTE
# =================================================================================
try:
    df_bronze_categorias = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_categoria").union(df_quarentena.select("id_categoria"))
    else:
        df_processados = df_silver_atual.select("id_categoria")
        
    df_micro_lote = df_bronze_categorias.join(df_processados, "id_categoria", "left_anti")
else:
    df_micro_lote = df_bronze_categorias

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de categorias para processar: {qtd_novos}")

# =================================================================================
# 2. LEITURA DAS TABELAS DE REFERÊNCIA (Auto-relação e Relação com Produtos Bronze)
# =================================================================================
# Referência completa da própria Bronze (para validar integridade da árvore e ciclos)
df_auto_ref = df_bronze_categorias \
    .select(
        F.col("id_categoria").cast("long").alias("id_cat_verificacao"),
        F.col("id_categoria_pai").cast("long").alias("id_pai_verificacao")
    ).dropDuplicates()

# CORREÇÃO R10: Ajustado de 5 para 12 conforme detectado no diagnóstico físico
QTD_RAIZ_ESPERADA = 12 

# CORREÇÃO R7: agora prioriza a Silver de produtos, caindo para a Bronze
# quando a Silver ainda não existir. Gera o schema (id_categoria_produto +
# tem_produto) que a célula seguinte (df_produtos_ref_blindado) espera —
# uma linha por categoria que tem ao menos 1 produto, com tem_produto=True;
# categorias sem produto ficam de fora e viram False depois do fillna.
if delta_existe("silver", "ecommerce_produtos", STORAGE_OPTIONS):
    print("Referência de ecommerce_produtos (R7 categorias): usando Silver.")
    df_produtos_bronze_ou_silver = ler_delta("silver", "ecommerce_produtos", STORAGE_OPTIONS)
elif delta_existe("bronze", "ecommerce_produtos", STORAGE_OPTIONS):
    print("Referência de ecommerce_produtos (R7 categorias): Silver ainda não existe — usando Bronze.")
    df_produtos_bronze_ou_silver = ler_delta("bronze", "ecommerce_produtos", STORAGE_OPTIONS)
else:
    print("Referência de ecommerce_produtos (R7 categorias): nem Silver nem Bronze encontradas. "
          "A Regra 7 tratará todas as subcategorias como sem produto.")
    df_produtos_bronze_ou_silver = None

if df_produtos_bronze_ou_silver is not None:
    df_produtos_ref = (
        df_produtos_bronze_ou_silver
        .select(F.col("id_categoria").cast("long").alias("id_categoria_produto"))
        .where(F.col("id_categoria_produto").isNotNull())
        .distinct()
        .withColumn("tem_produto", F.lit(True))
    )
else:
    df_produtos_ref = spark.createDataFrame([], "id_categoria_produto long, tem_produto boolean")

## Leitura da Bronze e Aplicação das 10 Regras de Data Quality



In [0]:
if qtd_novos > 0:
    # Proteção contra XSS (Regra 9)
    regex_xss = r"[<>&'\"/;]"
    
    # Janelas analíticas com ordenação interna para rastrear duplicados
    # Usamos o F.current_timestamp() ou qualquer coluna de ordenação (ex: dt_registro se houver)
    w_dedup_id = Window.partitionBy("id_categoria").orderBy(F.current_timestamp())
    w_dedup_nome = Window.partitionBy("id_categoria_pai", "nome_categoria").orderBy(F.current_timestamp())
    w_filhos = Window.partitionBy("id_categoria_pai")

    # 1. Cálculo da volumetria global de raízes (R10)
    total_raiz_data_lake = df_bronze_categorias \
        .filter(F.col("id_categoria_pai").isNull() | (F.trim(F.col("id_categoria_pai")) == "")) \
        .select(F.col("id_categoria").cast("long")).distinct().count()

    # 2. Mapeamento de pais com filhos (R8)
    df_progenitores = df_auto_ref \
        .filter(F.col("id_pai_verificacao").isNotNull()) \
        .select(F.col("id_pai_verificacao").cast("long").alias("id_pai_com_filhos")).distinct()

    # 3. Assegura o tipo da referência de produtos vindos da Bronze (R7)
    df_produtos_ref_blindado = df_produtos_ref \
        .select(F.col("id_categoria_produto").cast("long").alias("id_categoria_produto_clean"), F.col("tem_produto"))

    # 4. PREPARAÇÃO DO DATAFRAME COM NUMERAÇÃO DE LINHA PARA GARANTIR O PRIMEIRO SALVO
    df_base = (df_micro_lote
        .withColumn("id_categoria_num", F.col("id_categoria").cast("long"))
        .withColumn("id_categoria_pai_num", F.col("id_categoria_pai").cast("long"))
        .withColumn("nome_categoria_norm", F.trim(F.col("nome_categoria")))
        
        # Cria indexadores de linha dentro do lote
        .withColumn("row_id_cat", F.row_number().over(w_dedup_id))
        .withColumn("row_nome_nivel", F.row_number().over(w_dedup_nome))
        
        # Joins de validação hierárquica e integridade
        .join(
            df_auto_ref.select(F.col("id_cat_verificacao").cast("long").alias("id_pai_existe")), 
            F.col("id_categoria_pai").cast("long") == F.col("id_pai_existe"), 
            "left"
        )
        .join(
            df_auto_ref.select(
                F.col("id_cat_verificacao").cast("long").alias("id_pai_para_avo"),
                F.col("id_pai_verificacao").cast("long").alias("id_avo_existe")
            ),
            F.col("id_categoria_pai").cast("long") == F.col("id_pai_para_avo"),
            "left"
        )
        .join(df_progenitores, F.col("id_categoria").cast("long") == df_progenitores.id_pai_com_filhos, "left")
        .join(df_produtos_ref_blindado, F.col("id_categoria").cast("long") == df_produtos_ref_blindado.id_categoria_produto_clean, "left"))

    # Normalização de flags boleanas
    df_base = df_base.fillna({"tem_produto": False}).withColumn(
        "tem_filhos_bool", F.when(F.col("id_pai_com_filhos").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )

    # 5. APLICAÇÃO DAS 10 REGRAS REESTRUTURADAS
    df_silver_categorias = (df_base
        # CORREÇÃO R1: Nulo FALHA. Se for duplicado, o primeiro (row_id_cat == 1) SALVA, os outros falham
        .withColumn("r1_id_categoria_falhou", F.col("id_categoria_num").isNull() | (F.col("row_id_cat") > 1))
        .withColumn("r2_nome_categoria_falhou", F.col("nome_categoria_norm").isNull() | (F.col("nome_categoria_norm") == ""))
        .withColumn("r3_id_categoria_pai_fk_falhou", F.col("id_categoria_pai_num").isNotNull() & F.col("id_pai_existe").isNull())
        .withColumn("r4_ciclo_hierarquia_falhou", F.col("id_categoria_pai_num").isNotNull() & (F.col("id_categoria_num") == F.col("id_categoria_pai_num")))
        # CORREÇÃO R5: O primeiro nome do nível salva, as repetições idênticas sob o mesmo pai falham
        .withColumn("r5_nome_duplicado_nivel_falhou", F.col("row_nome_nivel") > 1)
        .withColumn("r6_nivel_hierarquia_excedido_falhou", F.col("id_categoria_pai_num").isNotNull() & F.col("id_avo_existe").isNotNull())
        .withColumn("r7_subcategoria_sem_produto_falhou", F.col("id_categoria_pai_num").isNotNull() & (F.col("tem_produto") == False))
        .withColumn("r8_raiz_sem_filhos_falhou", F.col("id_categoria_pai_num").isNull() & (F.col("tem_filhos_bool") == False))
        .withColumn("r9_caracteres_especiais_falhou", F.col("nome_categoria_norm").isNotNull() & F.col("nome_categoria_norm").rlike(regex_xss))
        .withColumn("r10_total_raiz_divergente_falhou", F.lit(total_raiz_data_lake != QTD_RAIZ_ESPERADA)))

    # =================================================================================
    # 📌 SISTEMA DE DIAGNÓSTICO ATUALIZADO
    # =================================================================================
    print(f"--- DETALHAMENTO DE FALHAS NO LOTE DE CATEGORIAS ---")
    print(f"Total de registros analisados no micro-lote: {df_silver_categorias.count()}")
    print(f"Quantidade de raízes detectadas no Data Lake (R10): {total_raiz_data_lake} (Esperado: {QTD_RAIZ_ESPERADA})")
    
    regras_verificacao = [
        ("R1 (ID Nulo/Duplicado)", "r1_id_categoria_falhou"),
        ("R2 (Nome Vazio)", "r2_nome_categoria_falhou"),
        ("R3 (Pai Inexistente)", "r3_id_categoria_pai_fk_falhou"),
        ("R4 (Ciclo Infinito)", "r4_ciclo_hierarquia_falhou"),
        ("R5 (Nome Duplicado Nível)", "r5_nome_duplicado_nivel_falhou"),
        ("R6 (Hierarquia > 2 Níveis)", "r6_nivel_hierarquia_excedido_falhou"),
        ("R7 (Subcategoria com Produto Bronze)", "r7_subcategoria_sem_produto_falhou"),
        ("R8 (Raiz sem Filhos)", "r8_raiz_sem_filhos_falhou"),
        ("R9 (Caracteres XSS)", "r9_caracteres_especiais_falhou"),
        ("R10 (Total Raiz Divergente)", "r10_total_raiz_divergente_falhou")
    ]
    
    for nome_regra, nome_coluna in regras_verificacao:
        falhas = df_silver_categorias.filter(F.col(nome_coluna) == True).count()
        print(f"⚠️ {nome_regra}: Encontrou {falhas} linhas com erro.")
    print(f"---------------------------------------------------")

    # Catálogo unificado para os logs
    catalogo_regras = [
        {"coluna": "r1_id_categoria_falhou", "regra": "R1_ID_CATEGORIA_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_nome_categoria_falhou", "regra": "R2_NOME_CATEGORIA_VAZIO", "severidade": "Critica"},
        {"coluna": "r3_id_categoria_pai_fk_falhou", "regra": "R3_PAI_INEXISTENTE_ORFAO", "severidade": "Critica"},
        {"coluna": "r4_ciclo_hierarquia_falhou", "regra": "R4_CICLO_AUTO_REFERENCIA", "severidade": "Critica"},
        {"coluna": "r5_nome_duplicado_nivel_falhou", "regra": "R5_AMBIGUIDADE_MESMO_NIVEL", "severidade": "Critica"},
        {"coluna": "r6_nivel_hierarquia_excedido_falhou", "regra": "R6_EXCEDEU_MAX_2_NIVEIS", "severidade": "Critica"},
        {"coluna": "r7_subcategoria_sem_produto_falhou", "regra": "R7_MENU_FANTASMA_SEM_PROD", "severidade": "Critica"},
        {"coluna": "r8_raiz_sem_filhos_falhou", "regra": "R8_RAIZ_INUTILIZAVEL_SEM_FILHO", "severidade": "Critica"},
        {"coluna": "r9_caracteres_especiais_falhou", "regra": "R9_VULNERABILIDADE_XSS_DETECTADA", "severidade": "Critica"},
        {"coluna": "r10_total_raiz_divergente_falhou", "regra": "R10_VOLUMETRIA_RAIZ_FORA_CONFIG", "severidade": "Critica"}
    ]

    total_registros = df_silver_categorias.count()
    logs_list = []
    for r in catalogo_regras:
        qtd_falhas = df_silver_categorias.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    # --- UNIFICAÇÃO FINAL DAS REGRAS ---
    todas_flags = [r["coluna"] for r in catalogo_regras]
    condicao_total_falha = reduce(lambda a, b: a | b, [F.col(c) for c in todas_flags])

    df_silver_categorias = (df_silver_categorias
        .withColumn("silver_linha_valida", ~condicao_total_falha)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))
    
    print("✓ Estrutura de qualidade reprocessada com salvamento do primeiro registro.")
else:
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Etapa ignorada: não há micro-lote novo.")

## Gravação Final (Silver Categorias e Logs)

In [0]:
# =================================================================================
# 3. GRAVAÇÃO BLINDADA VIA SDK DELTA
# =================================================================================
if qtd_novos > 0:
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]

    # A. Gravação de Registros Aprovados (Silver)
    df_silver_validos = (df_silver_categorias
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))

    qtd_validos = df_silver_validos.count()
    print(f"Categorias aprovadas para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
        )

    # B. Gravação de Rejeitados (Quarentena com Anti-Join por id_categoria)
    df_silver_invalidos = (df_silver_categorias
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))

    # CORREÇÃO: removido o "if" duplicado/mal indentado que quebrava a sintaxe
    if df_silver_invalidos.count() > 0:
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            # NULL nunca é igual a NULL num join — linhas com id_categoria
            # nulo (que a própria R1 marca como falha) nunca eram reconhecidas
            # como "já existentes" e voltavam a ser gravadas na quarentena a
            # cada execução, inflando a tabela sem limite. Deduplicamos por
            # id_categoria SOMENTE quando não é nulo; linhas com id nulo são
            # deduplicadas à parte, usando todas as colunas de negócio como
            # chave, para não acumular infinitamente.
            df_quarentena_com_id = df_silver_invalidos.filter(F.col("id_categoria").isNotNull())
            df_quarentena_sem_id = df_silver_invalidos.filter(F.col("id_categoria").isNull())

            df_novos_com_id = df_quarentena_com_id.join(
                df_quarentena_historico.select("id_categoria").where(F.col("id_categoria").isNotNull()),
                on="id_categoria",
                how="left_anti"
            )

            df_historico_sem_id = df_quarentena_historico.filter(F.col("id_categoria").isNull())
            colunas_comparacao = [c for c in df_quarentena_sem_id.columns if c not in ("silver_processed_at", "silver_run_id")]
            df_novos_sem_id = df_quarentena_sem_id.join(
                df_historico_sem_id.select(*colunas_comparacao).dropDuplicates(),
                on=colunas_comparacao,
                how="left_anti"
            )

            df_quarentena_para_gravar = df_novos_com_id.unionByName(df_novos_sem_id)
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            gravar_delta(
                df=df_quarentena_para_gravar, camada="silver/quarentena", tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
            )
            print(f"Enviados {qtd_novos_rejeitados} categorias novas para a quarentena.")

    # C. Gravação de Logs na Raiz — CORREÇÃO: essa seção tinha sumido da
    # célula, fazendo os logs de qualidade nunca serem persistidos fisicamente.
    if df_dq_monitoring_logs_novos.count() > 0:
        gravar_delta(
            df=df_dq_monitoring_logs_novos, camada="", tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
        )
        print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  Validação Final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"Aviso: tabela Silver {TABELA_ALVO} não encontrada.")


if delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print("Aviso: tabela dq_monitoring_logs não encontrada na raiz do Data Lake.")